# Trabalho 07 - Modelagem e Avaliação de Desempenho com SPN
**Contexto 1: Servidor Web**

### Lugares e Transições do Modelo
- `P_Chegada` (marcação inicial = `1`): Geração de requisições HTTP.
- `P_Fila` (marcação inicial = `0`): Requisição na entrada aguardando admissão.
- `P_TransCap` (marcação inicial = `C_FILA`): Vagas disponíveis na fila de espera.
- `P_Fila_Web` (marcação inicial = `0`): Fila FCFS do Servidor Web.
- `P_TransProc` (marcação inicial = `P_WS`): Núcleos livres nos servidores Web.
- `P_Trans` (marcação inicial = `0`): Requisições em processamento.
- `AD`: Transição temporizada exponencial (`AD = 1 / AR` ms).
- `T1` e `T2`: Transições imediatas de admissão na fila e alocação de servidor.
- `T_Trans`: Transição temporizada exponencial (`TS = 20` ms, *Infinite Server*).


In [1]:
import math
import random
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs('results', exist_ok=True)
os.makedirs('figures', exist_ok=True)
plt.rcParams['figure.dpi'] = 140
plt.rcParams['font.size'] = 10


## 1. Avaliação do Modelo SPN em Regime Estacionário

In [2]:
def evaluate_spn_steady_state(ar: float, p_ws: int, c_fila: int, ts: float = 20.0) -> dict:
    ad = 1.0 / ar
    mu = 1.0 / ts
    max_n = p_ws + c_fila

    log_weights = [0.0] * (max_n + 1)
    for n in range(1, max_n + 1):
        active_servers = min(n, p_ws)
        log_weights[n] = log_weights[n - 1] + math.log(ar / (active_servers * mu))

    max_log = max(log_weights)
    weights = [math.exp(lw - max_log) for lw in log_weights]
    total_w = sum(weights)
    pi = [w / total_w for w in weights]

    e_p_trans = sum(min(n, p_ws) * pi[n] for n in range(max_n + 1))
    e_p_fila_web = sum(max(0, n - p_ws) * pi[n] for n in range(max_n + 1))
    p_drop = pi[max_n]

    u_pct = (e_p_trans / p_ws) * 100.0
    dp_pct = p_drop * 100.0
    tp_ms = e_p_trans / ts
    tp_s = tp_ms * 1000.0
    mqt_ms = (e_p_fila_web * ad) / (1.0 - p_drop)
    mpt_ms = (e_p_trans * ad) / (1.0 - p_drop)
    mrt_ms = ((e_p_fila_web + e_p_trans) * ad) / (1.0 - p_drop)

    return {
        'AR': round(ar, 4),
        'AD': round(ad, 4),
        'P_WS': p_ws,
        'C_FILA': c_fila,
        'TS': round(ts, 2),
        'E_P_Fila_Web': round(e_p_fila_web, 4),
        'E_P_Trans': round(e_p_trans, 4),
        'U': round(u_pct, 2),
        'DP': round(dp_pct, 2),
        'TP_ms': round(tp_ms, 4),
        'TP_s': round(tp_s, 2),
        'MQT': round(mqt_ms, 2),
        'MPT': round(mpt_ms, 2),
        'MRT': round(mrt_ms, 2),
    }


## 2. Experimentos E1, E2, E3 e E4

In [3]:
TS_REF = 20.0
P_WS_REF = 8
C_FILA_REF = 20
AR_REF = 0.40

# E1: 5 níveis de taxa de chegada
e1_levels = [0.10, 0.25, 0.40, 0.55, 0.70]
e1_rows = [{'Experimento': f'E1.{i}', **evaluate_spn_steady_state(ar, P_WS_REF, C_FILA_REF, TS_REF)} for i, ar in enumerate(e1_levels, 1)]
pd.DataFrame(e1_rows).to_csv('results/E1_taxa_chegada.csv', index=False)

# E2: 3 configurações de capacidade de processamento
e2_rows = [{'Experimento': f'E2.{i}', **evaluate_spn_steady_state(AR_REF, p_ws, C_FILA_REF, TS_REF)} for i, p_ws in enumerate([4, 8, 12], 1)]
pd.DataFrame(e2_rows).to_csv('results/E2_capacidade_processamento.csv', index=False)

# E3: 3 limites de capacidade da fila
e3_rows = [{'Experimento': f'E3.{i}', **evaluate_spn_steady_state(AR_REF, P_WS_REF, cf, TS_REF)} for i, cf in enumerate([10, 20, 40], 1)]
pd.DataFrame(e3_rows).to_csv('results/E3_capacidade_fila.csv', index=False)

# E4: 3 cenários combinados de demanda e recursos
e4_specs = [(0.55, 6, 10), (0.55, 12, 20), (0.55, 16, 40)]
e4_rows = [{'Experimento': f'E4.{i}', **evaluate_spn_steady_state(ar, p_ws, cf, TS_REF)} for i, (ar, p_ws, cf) in enumerate(e4_specs, 1)]
pd.DataFrame(e4_rows).to_csv('results/E4_cenarios_combinados.csv', index=False)

df_consolidado = pd.DataFrame(e1_rows + e2_rows + e3_rows + e4_rows)
df_consolidado.to_csv('results/tabela_consolidada.csv', index=False)
print(df_consolidado[['Experimento', 'AR', 'P_WS', 'C_FILA', 'U', 'TP_s', 'MQT', 'MPT', 'MRT', 'DP']].to_string(index=False))


Experimento   AR  P_WS  C_FILA      U   TP_s   MQT   MPT    MRT    DP
       E1.1 0.10     8      20  25.00 100.00  0.00 20.00  20.00  0.00
       E1.2 0.25     8      20  62.50 250.00  1.11 20.00  21.11  0.00
       E1.3 0.40     8      20  95.88 383.50 22.59 20.00  42.59  4.12
       E1.4 0.55     8      20  99.98 399.93 43.37 20.00  63.37 27.29
       E1.5 0.70     8      20 100.00 400.00 46.67 20.00  66.67 42.86
       E2.1 0.40     4      20 100.00 200.00 95.00 20.00 115.00 50.00
       E2.2 0.40     8      20  95.88 383.50 22.59 20.00  42.59  4.12
       E2.3 0.40    12      20  66.67 399.99  0.70 20.00  20.70  0.00
       E3.1 0.40     8      10  92.98 371.92 10.38 20.00  30.38  7.02
       E3.2 0.40     8      20  95.88 383.50 22.59 20.00  42.59  4.12
       E3.3 0.40     8      40  97.74 390.96 47.40 20.00  67.40  2.26
       E4.1 0.55     6      10  99.97 299.92 29.36 20.00  49.36 45.47
       E4.2 0.55    12      20  90.62 543.72  8.41 20.00  28.41  1.14
       E4.3 0.55    

## 3. Gráficos dos Experimentos

In [4]:
ar_grid = np.linspace(0.05, 0.80, 16)

e1_curve = [evaluate_spn_steady_state(ar, P_WS_REF, C_FILA_REF, TS_REF) for ar in ar_grid]
fig, axs = plt.subplots(2, 2, figsize=(11, 8))
axs[0, 0].plot(ar_grid, [r['U'] for r in e1_curve], 'o-', color='#1f61b8', label='Utilização U (%)')
axs[0, 0].set_title('(a) Utilização dos Servidores (U)')
axs[0, 0].set_ylabel('Utilização (%)')
axs[0, 1].plot(ar_grid, [r['TP_s'] for r in e1_curve], 's-', color='#269947', label='Vazão Efetiva TP (req/s)')
axs[0, 1].set_title('(b) Vazão do Sistema (TP)')
axs[0, 1].set_ylabel('Vazão TP (req/s)')
axs[1, 0].plot(ar_grid, [r['MRT'] for r in e1_curve], 'o-', color='#d12e2e', label='Tempo Resp. (MRT)')
axs[1, 0].plot(ar_grid, [r['MQT'] for r in e1_curve], 's-', color='#e07a14', label='Tempo Fila (MQT)')
axs[1, 0].plot(ar_grid, [r['MPT'] for r in e1_curve], '^-', color='#1f61b8', label='Tempo Proc. (MPT)')
axs[1, 0].set_title('(c) Tempos Médios (MRT, MQT, MPT)')
axs[1, 0].set_ylabel('Tempo (ms)')
axs[1, 1].plot(ar_grid, [r['DP'] for r in e1_curve], '^-', color='#d12e2e', label='Prob. Descarte DP (%)')
axs[1, 1].set_title('(d) Probabilidade de Descarte (DP)')
axs[1, 1].set_ylabel('Prob. de Descarte (%)')
for ax in axs.flat:
    ax.set_xlabel('Taxa de chegada AR (req/ms)')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()
plt.tight_layout()
plt.savefig('figures/fig_e1_demanda.png', dpi=300)
plt.show()


In [5]:
e2_curves = {p: [evaluate_spn_steady_state(ar, p, C_FILA_REF, TS_REF) for ar in ar_grid] for p in [4, 8, 12]}
fig, axs = plt.subplots(2, 2, figsize=(11, 8))
colors = {4: ('#d12e2e', 'o-', 'Config. A (P_WS=4)'), 8: ('#1f61b8', 's-', 'Config. B (P_WS=8)'), 12: ('#269947', '^-', 'Config. C (P_WS=12)')}
for p, (col, mk, lbl) in colors.items():
    axs[0, 0].plot(ar_grid, [r['U'] for r in e2_curves[p]], mk, color=col, label=lbl)
    axs[0, 1].plot(ar_grid, [r['TP_s'] for r in e2_curves[p]], mk, color=col, label=lbl)
    axs[1, 0].plot(ar_grid, [r['MRT'] for r in e2_curves[p]], mk, color=col, label=lbl)
    axs[1, 1].plot(ar_grid, [r['DP'] for r in e2_curves[p]], mk, color=col, label=lbl)
axs[0, 0].set_title('(a) Utilização por Capacidade (U)'); axs[0, 0].set_ylabel('Utilização (%)')
axs[0, 1].set_title('(b) Vazão por Capacidade (TP)'); axs[0, 1].set_ylabel('Vazão TP (req/s)')
axs[1, 0].set_title('(c) Tempo Médio de Resposta (MRT)'); axs[1, 0].set_ylabel('MRT (ms)')
axs[1, 1].set_title('(d) Probabilidade de Descarte (DP)'); axs[1, 1].set_ylabel('Prob. de Descarte (%)')
for ax in axs.flat:
    ax.set_xlabel('Taxa de chegada AR (req/ms)')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()
plt.tight_layout()
plt.savefig('figures/fig_e2_capacidade.png', dpi=300)
plt.show()


In [6]:
e3_curves = {cf: [evaluate_spn_steady_state(ar, P_WS_REF, cf, TS_REF) for ar in ar_grid] for cf in [10, 20, 40]}
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
for cf, col, mk in [(10, '#269947', 'o-'), (20, '#1f61b8', 's-'), (40, '#d12e2e', '^-')]:
    axs[0].plot(ar_grid, [r['MRT'] for r in e3_curves[cf]], mk, color=col, label=f'C_FILA = {cf}')
    axs[1].plot(ar_grid, [r['DP'] for r in e3_curves[cf]], mk, color=col, label=f'C_FILA = {cf}')
axs[0].set_title('(a) Impacto de C_FILA no MRT'); axs[0].set_ylabel('MRT (ms)')
axs[1].set_title('(b) Impacto de C_FILA no Descarte (DP)'); axs[1].set_ylabel('Prob. de Descarte (%)')
for ax in axs:
    ax.set_xlabel('Taxa de chegada AR (req/ms)')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()
plt.tight_layout()
plt.savefig('figures/fig_e3_fila.png', dpi=300)
plt.show()


In [7]:
labels_c = ['C1 (P_WS=6)', 'C2 (P_WS=12)', 'C3 (P_WS=16)']
x = np.arange(len(labels_c))
width = 0.34

fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
axs[0].bar(x - width/2, [r['MRT'] for r in e4_rows], width, label='MRT (ms)', color='#266bc0')
axs[0].bar(x + width/2, [r['MQT'] for r in e4_rows], width, label='MQT (ms)', color='#e6801f')
axs[0].set_xticks(x); axs[0].set_xticklabels(labels_c)
axs[0].set_ylabel('Tempo (ms)'); axs[0].set_title('(a) Tempo de Resposta (MRT) e Fila (MQT)')
axs[0].grid(True, axis='y', linestyle='--', alpha=0.5); axs[0].legend()

axs[1].bar(x - width/2, [r['U'] for r in e4_rows], width, label='Utilização U (%)', color='#2e9e4c')
axs[1].bar(x + width/2, [r['DP'] for r in e4_rows], width, label='Descarte DP (%)', color='#d12e2e')
axs[1].set_xticks(x); axs[1].set_xticklabels(labels_c)
axs[1].set_ylabel('Percentual (%)'); axs[1].set_title('(b) Utilização (U) e Prob. de Descarte (DP)')
axs[1].grid(True, axis='y', linestyle='--', alpha=0.5); axs[1].legend()

plt.tight_layout()
plt.savefig('figures/fig_e4_combinado.png', dpi=300)
plt.show()
